## Normalização de dados

In [0]:
%sql
-- Objetivo: Limpar, remover duplicata, mantenho a tabela raw e criar uma nova tabela pra manter
-- com os dados normalizados e estruturar a tabela de produtos (Camada Prata).
-- ==============================================================================

CREATE OR REPLACE TABLE lh_nautical.lh_nautical_db.slv_produtos AS
WITH cte_produtos_deduplicados AS (
    -- Filtro de nulos e duplicatas
    SELECT DISTINCT * FROM lh_nautical.lh_nautical_db.produtos_raw
    WHERE code IS NOT NULL 
)
-- Transformação e estruturação
SELECT 
    code AS id_product,
    name AS original_name,
    
    -- Extrai do name, o tipo do produto
    -- informação que pode ser interessante a ser utilizada posteriormente
    SPLIT(name, ' ')[0] AS product_type,
    
    -- Usando TRY_CAST para evitar erros de conversão de string para int ao 
    -- capturar valor da potencia do produto do tipo motor
    TRY_CAST(REGEXP_EXTRACT(name, '([0-9]+)HP', 1) AS INT) AS horsepower,
    
    -- Padronização das categorias, que estavam com descrição diferentes
    -- e com acentos, por exemplo
    -- e também cria uma categoria para os produtos que não tem categoria
    CASE 
        WHEN UPPER(REPLACE(actual_category, ' ', '')) LIKE 'ELET%' THEN 'ELETRONICOS'
        WHEN UPPER(REPLACE(actual_category, ' ', '')) LIKE 'PROP%' THEN 'PROPULSAO'
        WHEN UPPER(REPLACE(actual_category, ' ', '')) LIKE '%NCOR%' THEN 'ANCORAGEM'
        ELSE UPPER(TRIM(actual_category)) 
    END AS category_clean,
    
    -- Limpeza do Preço para que seja um valor numérico
    -- e remoção de caracteres especiais, espaços e coloca o ponto no lugar da virgula
    -- seguindo o formato padrao americano
    TRY_CAST(REPLACE(REPLACE(price, 'R$ ', ''), ',', '.') AS DOUBLE) AS price_num
    
FROM cte_produtos_deduplicados;

-- Amostra para vlidação de que tudo funcionou corretamente
SELECT * FROM lh_nautical.lh_nautical_db.slv_produtos LIMIT 10;

id_product,original_name,product_type,horsepower,category_clean,price_num
24,GPS AIS Orca Marlin Velocity,GPS,null,ELETRONICOS,23232.18
40,Sonda Furuno Maré Tidal,Sonda,null,ELETRONICOS,14184.04
46,Sonda AIS Magnum Boost,Sonda,null,ELETRONICOS,4983.77
60,Motor de Popa Honda Velocity Drift 258HP,Motor,258,PROPULSAO,108751.09
105,Cabo de Nylon Danforth Prime,Cabo,null,ANCORAGEM,309.54
116,Cabo de Nylon Delta Velocity Oceanic Abyss,Cabo,null,ANCORAGEM,2575.52
145,Boia de Arqueamento Delta Nexus,Boia,null,ANCORAGEM,4349.86
130,Boia de Arqueamento Delta Zen Boost Vortex,Boia,null,ANCORAGEM,4144.09
147,Corrente Danforth Force Leviathan Impulse,Corrente,null,ANCORAGEM,3030.08
52,Motor Diesel Honda Zen Tidal Mako 26HP,Motor,26,PROPULSAO,13704.1


## Consultas gerais

In [0]:
%sql
-- Consultas aleatória para nalise expliratória dos registros

-- SELECT DISTINCT * FROM lh_nautical.lh_nautical_db.slv_produtos order by id_product
-- SELECT * FROM lh_nautical.lh_nautical_db.vendas_2023_2024 LIMIT 50
-- DESCRIBE lh_nautical.lh_nautical_db.slv_produtos

id,id_client,id_product,qtd,total,sale_date
0,42,105,11,3405.0,2023-09-10
1,3,136,9,16873.9,15-09-2024
2,25,139,7,9475.3,2024-08-13
4,20,23,5,55893.0,2023-02-03
5,8,57,4,451403.9,2024-02-12
6,36,52,3,39056.4,2023-09-26
8,27,25,3,34560.05,2024-02-28
9,37,26,7,114932.9,07-11-2023
10,31,143,3,12643.55,2024-08-25
11,39,128,5,23254.0,2023-05-07
